# Member 1 — eBird ingestion & cleaning (AviTrend PH)

Turns raw eBird downloads into analysis-ready tables that separate **true absences** from **observation gaps**.
Follows the *Member 1* section of `docs/TEAM_BUILD_PLAN.md`.

| Term | Meaning | Stored as |
|---|---|---|
| Detection | A usable complete checklist reported the species | `detected = 1` |
| True absence | A usable complete checklist did **not** report the species | `detected = 0` |
| Observation gap | No usable checklist in that cell and month | no row; `is_gap = True` in the effort grid |
| Low-data cell | Fewer than 5 usable checklists in the cell over the whole period | `low_data = True` |

**Inputs:** eBird Basic Dataset zips *with Sampling Event Data* (`ebd_PH_..._smp_....zip`) in `Raw EBIRD Records/`, and the Avilight species masterlist.

**Outputs** (`data/processed/`): `checklists`, `detections`, `species`, `effort_grid` (Parquet, or CSV if `pyarrow` is missing), `filter_log.csv`, `species_unmatched.csv`, `figures/hotspots.png`.

**Run order:** top to bottom. Edit only the **Config** cell.

## 0. Setup and config

In [ ]:
import sys, os, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("matplotlib not installed: the hotspot figure will be skipped")

try:
    import pyarrow  # noqa: F401
    HAS_PARQUET = True
except ImportError:
    HAS_PARQUET = False
    print("pyarrow not installed: tables will be written as CSV instead of Parquet")

IN_COLAB = "google.colab" in sys.modules

In [ ]:
# ===================== CONFIG (edit here only) =====================
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/NASA Hackathon")   # <- folder that holds 'Raw EBIRD Records'
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = PROJECT_ROOT / "Raw EBIRD Records"
OUT_DIR = PROJECT_ROOT / "data" / "processed"

# Analysis window (plan: Jan 2014 - Dec 2025)
DATE_START, DATE_END = "2014-01-01", "2025-12-31"

# Greater Manila focus box and 1 km grid (must match Member 2's grid)
LAT_MIN, LAT_MAX = 14.00, 15.30
LON_MIN, LON_MAX = 120.50, 121.60
CELL_DEG = 0.009

# eBird Best Practices filters
PROTOCOLS = ["Stationary", "Traveling"]
DUR_MIN, DUR_MAX = 5, 300          # minutes
DIST_MAX_KM = 5.0                  # main bird model later uses <= 2 km (flag column `within_2km`)
MAX_OBSERVERS = 10
KEEP_CATEGORIES = ["species", "issf", "form", "domestic"]   # COMMON NAME is already species-level for these

# Species masterlist (Avilight). A URL or a local CSV path. None = run without traits (testing only).
MASTERLIST_SOURCE = "https://docs.google.com/spreadsheets/d/1k4oiChJPhkroGhKxU7gX2ngxLTs6Z-9jSWiCBpUINCw/export?format=csv"
ML_NAME_COL, ML_TOL_COL, ML_MIG_COL = "Species Name", "Light Tolerance", "Migratory Status"

# Focus species: detected on >= FOCUS_MIN_DETECTIONS checklists AND on >= FOCUS_MIN_RATE of all checklists,
# AND (if REQUIRE_TRAITS) with a known light tolerance.
FOCUS_MIN_DETECTIONS = 50
FOCUS_MIN_RATE = 0.01
REQUIRE_TRAITS = True

LOW_DATA_MIN = 5                   # checklists per cell over the whole period
# ===================================================================

## 1. Helpers

In [ ]:
N_ROWS = int(np.ceil(round((LAT_MAX - LAT_MIN) / CELL_DEG, 6)))
N_COLS = int(np.ceil(round((LON_MAX - LON_MIN) / CELL_DEG, 6)))

def grid_rc(lat, lon):
    lat = np.asarray(lat, dtype=float); lon = np.asarray(lon, dtype=float)
    return (np.floor((lat - LAT_MIN) / CELL_DEG).astype(int),
            np.floor((lon - LON_MIN) / CELL_DEG).astype(int))

def cell_id_from_rc(row, col):
    return pd.Series(row).map("{:03d}".format).values + "_" + pd.Series(col).map("{:03d}".format).values

def write_table(df, name):
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    path = OUT_DIR / (f"{name}.parquet" if HAS_PARQUET else f"{name}.csv")
    if HAS_PARQUET:
        df.to_parquet(path, index=False)
    else:
        df.to_csv(path, index=False)
    print(f"wrote {path.name:28s} {len(df):>12,} rows")
    return path

filter_log = []
def log_step(step, df):
    filter_log.append({"step": step, "checklists_remaining": int(len(df))})
    print(f"{step:42s}{len(df):>10,}")

READ = dict(sep="\t", quoting=3, low_memory=False, dtype=str)   # quoting=3: eBird text fields contain quotes

def ebird_members(zpath):
    with zipfile.ZipFile(zpath) as z:
        names = z.namelist()
    samp = [n for n in names if n.endswith("_sampling.txt")]
    obs = [n for n in names if n.startswith("ebd_") and n.endswith(".txt") and not n.endswith("_sampling.txt")]
    return (samp[0] if samp else None), (obs[0] if obs else None)

print(f"grid: {N_ROWS} rows x {N_COLS} cols = {N_ROWS * N_COLS:,} cells")

## 2. Load the sampling files (one row per checklist) and remove duplicates

Downloads can overlap (e.g. a Jan–Dec file and two half-year files). Each checklist is assigned to the **first** zip that contains it, so its observations are read exactly once.

In [ ]:
SAMPLING_COLS = ["SAMPLING EVENT IDENTIFIER", "GROUP IDENTIFIER", "OBSERVATION DATE", "TIME OBSERVATIONS STARTED",
                 "PROTOCOL NAME", "PROTOCOL TYPE", "DURATION MINUTES", "EFFORT DISTANCE KM", "NUMBER OBSERVERS",
                 "ALL SPECIES REPORTED", "LATITUDE", "LONGITUDE", "LOCALITY", "LOCALITY ID", "LOCALITY TYPE"]

zips = sorted(RAW_DIR.glob("ebd_*.zip"))
assert zips, f"No eBird zips found in {RAW_DIR}"

parts = []
for zp in zips:
    samp_name, obs_name = ebird_members(zp)
    if samp_name is None:
        warnings.warn(f"{zp.name} has no sampling file - skipped (download WITH sampling event data)")
        continue
    with zipfile.ZipFile(zp) as z:
        s = pd.read_csv(z.open(samp_name), usecols=lambda c: c in SAMPLING_COLS, **READ)
    if "PROTOCOL TYPE" in s.columns and "PROTOCOL NAME" not in s.columns:   # older releases
        s = s.rename(columns={"PROTOCOL TYPE": "PROTOCOL NAME"})
    s["source_zip"] = zp.name
    parts.append(s)
    print(f"{zp.name:50s}{len(s):>9,} checklists")

samp = pd.concat(parts, ignore_index=True)
log_step("all checklists in downloads (with overlaps)", samp)
samp = samp.drop_duplicates("SAMPLING EVENT IDENTIFIER", keep="first")
log_step("unique checklists", samp)

## 3. Crop to the window and the Greater Manila box, then apply the eBird Best Practices filters

In [ ]:
c = samp.copy()
c["date"] = pd.to_datetime(c["OBSERVATION DATE"], errors="coerce")
for col in ["LATITUDE", "LONGITUDE", "DURATION MINUTES", "EFFORT DISTANCE KM", "NUMBER OBSERVERS", "ALL SPECIES REPORTED"]:
    c[col] = pd.to_numeric(c[col], errors="coerce")

c = c[c["date"].between(DATE_START, DATE_END)];                                   log_step(f"in window {DATE_START}..{DATE_END}", c)
c = c[c["LATITUDE"].between(LAT_MIN, LAT_MAX) & c["LONGITUDE"].between(LON_MIN, LON_MAX)]
log_step("inside Greater Manila box", c)
c = c[c["ALL SPECIES REPORTED"] == 1];                                            log_step("complete checklists", c)
c = c[c["PROTOCOL NAME"].isin(PROTOCOLS)];                                        log_step("Stationary or Traveling", c)
c = c[c["DURATION MINUTES"].between(DUR_MIN, DUR_MAX)];                           log_step(f"duration {DUR_MIN}-{DUR_MAX} min", c)
c["distance_km"] = np.where(c["PROTOCOL NAME"] == "Stationary", 0.0, c["EFFORT DISTANCE KM"])
c = c[c["distance_km"] <= DIST_MAX_KM];                                           log_step(f"distance <= {DIST_MAX_KM:g} km", c)
c = c[c["NUMBER OBSERVERS"].between(1, MAX_OBSERVERS)];                           log_step(f"observers <= {MAX_OBSERVERS}", c)

## 4. Collapse shared checklists

When people bird together, eBird stores one copy per person under the same `GROUP IDENTIFIER`. Keep one per group (lowest checklist ID).

In [ ]:
grp = c["GROUP IDENTIFIER"].notna() & (c["GROUP IDENTIFIER"] != "")
c = pd.concat([c[~grp],
               c[grp].sort_values("SAMPLING EVENT IDENTIFIER").drop_duplicates("GROUP IDENTIFIER", keep="first")],
              ignore_index=True)
log_step("one checklist per shared group", c)

## 5. Derived columns: time, effort and grid cell

In [ ]:
c["year"] = c["date"].dt.year.astype(int)
c["month"] = c["date"].dt.month.astype(int)
c["week"] = ((c["date"].dt.dayofyear - 1) // 7 + 1).clip(upper=52).astype(int)     # 52 weeks; day 365/366 -> week 52
t = c["TIME OBSERVATIONS STARTED"].str.split(":", expand=True)
c["hour_started"] = pd.to_numeric(t[0], errors="coerce") + pd.to_numeric(t[1], errors="coerce") / 60
row, col = grid_rc(c["LATITUDE"], c["LONGITUDE"])
c["cell_row"], c["cell_col"] = row, col
c["cell_id"] = cell_id_from_rc(row, col)
c["within_2km"] = c["distance_km"] <= 2.0
c = c.rename(columns={"SAMPLING EVENT IDENTIFIER": "checklist_id", "PROTOCOL NAME": "protocol",
                      "DURATION MINUTES": "duration_min", "NUMBER OBSERVERS": "n_observers",
                      "LATITUDE": "lat", "LONGITUDE": "lon", "LOCALITY": "locality",
                      "LOCALITY ID": "locality_id", "LOCALITY TYPE": "locality_type"})
print(c[["checklist_id", "date", "protocol", "duration_min", "distance_km", "cell_id"]].head())

## 6. Load observations for the kept checklists

Approved records only. Subspecies and forms (`issf`, `form`, `domestic`) count as their species; `spuh`, `slash` and `hybrid` are dropped because they cannot be assigned to one species. A species can appear on several rows of one checklist (e.g. species + subspecies), so rows are combined per checklist and species.

In [ ]:
OBS_COLS = ["SAMPLING EVENT IDENTIFIER", "CATEGORY", "COMMON NAME", "SCIENTIFIC NAME", "OBSERVATION COUNT", "APPROVED"]
ids_by_zip = c.groupby("source_zip")["checklist_id"].apply(set).to_dict()

obs_parts = []
for zp in zips:
    ids = ids_by_zip.get(zp.name)
    if not ids:
        continue
    _, obs_name = ebird_members(zp)
    with zipfile.ZipFile(zp) as z:
        for ch in pd.read_csv(z.open(obs_name), usecols=lambda col: col in OBS_COLS, chunksize=300_000, **READ):
            ch = ch[ch["SAMPLING EVENT IDENTIFIER"].isin(ids)]
            if "APPROVED" in ch.columns:
                ch = ch[ch["APPROVED"] == "1"]
            obs_parts.append(ch[ch["CATEGORY"].isin(KEEP_CATEGORIES)])
    print(f"read observations from {zp.name}")

obs = pd.concat(obs_parts, ignore_index=True).rename(columns={
    "SAMPLING EVENT IDENTIFIER": "checklist_id", "COMMON NAME": "species", "SCIENTIFIC NAME": "scientific_name"})
obs["count_num"] = pd.to_numeric(obs["OBSERVATION COUNT"], errors="coerce")     # 'X' -> NaN
obs["is_x"] = obs["count_num"].isna()

det = (obs.groupby(["checklist_id", "species"], as_index=False)
          .agg(count=("count_num", lambda s: s.sum(min_count=1)), any_x=("is_x", "max"),
               scientific_name=("scientific_name", "first")))
det.loc[det["any_x"], "count"] = np.nan       # count unknown if any row was 'X'; detection is still 1
det = det.drop(columns="any_x")
print(f"{len(obs):,} observation rows -> {len(det):,} checklist x species detections")

## 7. Drop checklists with no bird records

A complete 5–300 minute checklist with zero birds is implausible here: most are data-entry errors, test entries, or checklists whose only records were withheld (sensitive species). They are excluded (<1% of checklists) rather than counted as absences for every species.

In [ ]:
has_birds = c["checklist_id"].isin(det["checklist_id"])
print(f"checklists with no bird records: {(~has_birds).sum():,} ({(~has_birds).mean():.2%})")
c = c[has_birds].copy()
log_step("has at least one bird record (final)", c)

## 8. Species list, masterlist traits and focus species

In [ ]:
n_chk = len(c)
species = (det.groupby("species", as_index=False)
              .agg(scientific_name=("scientific_name", "first"), n_detections=("checklist_id", "nunique")))
species["reporting_rate"] = species["n_detections"] / n_chk

def norm(s):
    return s.astype(str).str.strip().str.lower().str.replace(r"\s+", " ", regex=True).str.replace("’", "'")

if MASTERLIST_SOURCE:
    ml = pd.read_csv(MASTERLIST_SOURCE)
    ml = ml[[ML_NAME_COL, ML_TOL_COL, ML_MIG_COL]].dropna(subset=[ML_NAME_COL])
    ml["key"] = norm(ml[ML_NAME_COL])
    ml = ml.drop_duplicates("key").rename(columns={ML_TOL_COL: "light_tolerance", ML_MIG_COL: "migratory_status"})
    species["key"] = norm(species["species"])
    species = species.merge(ml[["key", "light_tolerance", "migratory_status"]], on="key", how="left").drop(columns="key")
else:
    warnings.warn("MASTERLIST_SOURCE is None: no traits joined (testing only)")
    species["light_tolerance"] = np.nan
    species["migratory_status"] = np.nan

threshold = max(FOCUS_MIN_DETECTIONS, int(np.ceil(FOCUS_MIN_RATE * n_chk)))
common = species["n_detections"] >= threshold
has_trait = species["light_tolerance"].notna()
species["focus"] = common & has_trait if REQUIRE_TRAITS else common

unmatched = species[common & ~has_trait].sort_values("n_detections", ascending=False)
OUT_DIR.mkdir(parents=True, exist_ok=True)
unmatched[["species", "scientific_name", "n_detections"]].to_csv(OUT_DIR / "species_unmatched.csv", index=False)

print(f"species detected: {len(species):,} | threshold: >= {threshold:,} checklists")
print(f"common enough: {common.sum():,} | with trait: {(common & has_trait).sum():,} | focus: {species['focus'].sum():,}")
print(f"common species missing a trait (fix names in the masterlist): {len(unmatched):,} -> species_unmatched.csv")

## 9. Zero-fill: every usable checklist × every focus species

`detected = 1` if reported, else `0` (true absence). No median imputation of `X`: for detection it is simply 1, and `count` is left blank.

In [ ]:
focus_list = species.loc[species["focus"], "species"].sort_values().tolist()
grid_idx = pd.MultiIndex.from_product([c["checklist_id"].values, focus_list], names=["checklist_id", "species"])
detections = (pd.DataFrame(index=grid_idx).reset_index()
                .merge(det[det["species"].isin(focus_list)][["checklist_id", "species", "count"]],
                       on=["checklist_id", "species"], how="left", indicator=True))
detections["detected"] = (detections.pop("_merge") == "both").astype("int8")
detections["species"] = detections["species"].astype("category")
detections = detections[["checklist_id", "species", "detected", "count"]]
print(f"{len(detections):,} rows = {n_chk:,} checklists x {len(focus_list):,} focus species")
print(f"overall detection share: {detections['detected'].mean():.3f}")

## 10. Effort grid: every cell × month, including empty ones

`is_gap` = no usable checklist that month. `low_data` = fewer than 5 usable checklists in the cell over the whole period. Water cells are flagged later by Member 2 (`water_frac`).

In [ ]:
months = pd.period_range(DATE_START, DATE_END, freq="M")
rr, cc_ = np.meshgrid(np.arange(N_ROWS), np.arange(N_COLS), indexing="ij")
cells = pd.DataFrame({"cell_row": rr.ravel(), "cell_col": cc_.ravel()})
cells["cell_id"] = cell_id_from_rc(cells["cell_row"].values, cells["cell_col"].values)
ym = pd.DataFrame({"year": months.year, "month": months.month})
effort = cells.merge(ym, how="cross")

counts = (c.assign(hours=c["duration_min"] / 60)
           .groupby(["cell_id", "year", "month"], as_index=False)
           .agg(n_checklists=("checklist_id", "size"), total_hours=("hours", "sum")))
effort = effort.merge(counts, on=["cell_id", "year", "month"], how="left")
effort[["n_checklists", "total_hours"]] = effort[["n_checklists", "total_hours"]].fillna(0)
effort["n_checklists"] = effort["n_checklists"].astype(int)
effort["is_gap"] = effort["n_checklists"] == 0
cell_tot = effort.groupby("cell_id")["n_checklists"].transform("sum")
effort["low_data"] = (cell_tot > 0) & (cell_tot < LOW_DATA_MIN)
print(f"cells with >= 1 checklist: {(cell_tot.groupby(effort['cell_id']).first() > 0).sum():,} of {len(cells):,}")
print(f"cell-months that are gaps: {effort['is_gap'].mean():.1%}")

## 11. Save outputs

In [ ]:
checklists = c[["checklist_id", "date", "year", "month", "week", "hour_started", "protocol", "duration_min",
                "distance_km", "within_2km", "n_observers", "lat", "lon", "cell_row", "cell_col", "cell_id",
                "locality", "locality_id", "locality_type"]].sort_values("checklist_id").reset_index(drop=True)
species_out = species[["species", "scientific_name", "n_detections", "reporting_rate",
                       "light_tolerance", "migratory_status", "focus"]].sort_values("n_detections", ascending=False)

write_table(checklists, "checklists")
write_table(detections, "detections")
write_table(species_out, "species")
write_table(effort, "effort_grid")
pd.DataFrame(filter_log).to_csv(OUT_DIR / "filter_log.csv", index=False)
print("wrote filter_log.csv")

## 12. Checks (all must pass before handing off)

In [ ]:
results = []
def check(name, ok, detail=""):
    results.append((name, bool(ok)))
    print(f"[{'PASS' if ok else 'FAIL'}] {name} {detail}")

check("checklist_id is unique", checklists["checklist_id"].is_unique)
check("detections rows = checklists x focus species", len(detections) == len(checklists) * len(focus_list),
      f"({len(detections):,} vs {len(checklists):,} x {len(focus_list)})")
check("every detection's checklist exists", detections["checklist_id"].isin(checklists["checklist_id"]).all())
check("filters hold (duration, distance, observers)",
      checklists["duration_min"].between(DUR_MIN, DUR_MAX).all() and (checklists["distance_km"] <= DIST_MAX_KM).all()
      and (checklists["n_observers"] <= MAX_OBSERVERS).all())
check("all inside the focus box",
      checklists["lat"].between(LAT_MIN, LAT_MAX).all() and checklists["lon"].between(LON_MIN, LON_MAX).all())
check("effort grid has cells x months rows", len(effort) == N_ROWS * N_COLS * len(months),
      f"({len(effort):,} = {N_ROWS * N_COLS:,} x {len(months)})")
check("effort grid total = number of checklists", effort["n_checklists"].sum() == len(checklists))
check("UP Diliman cell id is 072_062", cell_id_from_rc(*grid_rc([14.655], [121.065]))[0] == "072_062")

fl = pd.DataFrame(filter_log)
fl["kept_share"] = fl["checklists_remaining"] / fl["checklists_remaining"].shift(1)
big_loss = fl[(fl["kept_share"] < 0.10) & ~fl["step"].str.startswith(("in window", "inside"))]
check("no filter step loses > 90%", big_loss.empty, "" if big_loss.empty else big_loss["step"].tolist())
print(fl.to_string(index=False))

### Spot check: 3 species recounted straight from the raw files

An independent pass: counts checklists reporting each species directly in the raw observation files (restricted to the final checklists) and compares with `detected.sum()`.

In [ ]:
spot = species_out.loc[species_out["focus"], "species"].head(3).tolist() or species_out["species"].head(3).tolist()
final_ids = set(checklists["checklist_id"])
raw_ids = {s: set() for s in spot}
for zp in zips:
    _, obs_name = ebird_members(zp)
    with zipfile.ZipFile(zp) as z:
        for ch in pd.read_csv(z.open(obs_name), usecols=["SAMPLING EVENT IDENTIFIER", "COMMON NAME", "CATEGORY", "APPROVED"],
                              chunksize=300_000, **READ):
            ch = ch[ch["COMMON NAME"].isin(spot) & ch["CATEGORY"].isin(KEEP_CATEGORIES) & (ch["APPROVED"] == "1")]
            for s, g in ch.groupby("COMMON NAME"):
                raw_ids[s] |= set(g["SAMPLING EVENT IDENTIFIER"]) & final_ids
for s in spot:
    pipe = int(detections.loc[detections["species"] == s, "detected"].sum()) if s in focus_list \
        else int(det.loc[det["species"] == s, "checklist_id"].nunique())
    check(f"spot check {s}", pipe == len(raw_ids[s]), f"(pipeline {pipe:,} vs raw {len(raw_ids[s]):,})")

### Plausibility: most-reported species and hotspot map

The top species should be common urban birds (e.g. Yellow-vented Bulbul, Eurasian Tree Sparrow, Zebra Dove). Known hotspots should light up: La Mesa / UP Diliman area, Rizal Park area, Las Piñas–Parañaque wetlands, Candaba marsh, Mt. Makiling / Los Baños. Most cells are gaps.

In [ ]:
print(species_out.head(10)[["species", "n_detections", "reporting_rate", "light_tolerance", "focus"]].to_string(index=False))

tot = effort.groupby(["cell_row", "cell_col"])["n_checklists"].sum().unstack().reindex(
    index=range(N_ROWS), columns=range(N_COLS)).fillna(0).values
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(7, 8))
    img = np.where(tot > 0, np.log10(tot), np.nan)
    im = ax.imshow(img, origin="lower", extent=[LON_MIN, LON_MIN + N_COLS * CELL_DEG, LAT_MIN, LAT_MIN + N_ROWS * CELL_DEG],
                   cmap="viridis")
    ax.set_facecolor("#e6e6e6")
    fig.colorbar(im, ax=ax, label="log10(usable checklists per 1 km cell)")
    ax.set_title(f"eBird effort {DATE_START[:4]}-{DATE_END[:4]} (grey = observation gap)")
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    (OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
    fig.savefig(OUT_DIR / "figures" / "hotspots.png", dpi=150, bbox_inches="tight")
    plt.show()
top = effort.groupby("cell_id")["n_checklists"].sum().sort_values(ascending=False).head(5)
print("busiest cells:", top.to_dict())

## 13. Hand-off summary (paste into the team channel)

In [ ]:
n_fail = sum(not ok for _, ok in results)
print(f"Checks: {len(results) - n_fail}/{len(results)} passed" + ("" if n_fail == 0 else "  <-- fix before handing off"))
print(f"Window {DATE_START}..{DATE_END} | usable checklists: {len(checklists):,} "
      f"(<= 2 km: {checklists['within_2km'].sum():,}) | focus species: {len(focus_list)} | "
      f"cells with data: {(tot > 0).sum():,} of {N_ROWS * N_COLS:,}")
print("Files in", OUT_DIR)